In [1]:
print("")

In [18]:
from langchain_core.tools import InjectedToolArg
from langchain_core.tools import tool
from typing import Annotated
import requests
import json
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
load_dotenv()
llm = ChatOpenAI(model = 'gpt-5')
@tool
def get_conversion_factor(base_currency:str, target_currency : str) -> float:
    '''This function fetches the currency conversion factor between a given base currency and target currency'''
    url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'
    response = requests.get(url)
    return response.json()

@tool
def convert(base_currency : int, conversion_rate : Annotated[float, InjectedToolArg]) -> float:
    '''Given a currency conversion rate this function calculates the target currency value from the given base currency value'''
    return base_currency * conversion_rate

llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [26]:
messages = [HumanMessage(content = "What is the conversion factor between INR and USD , and based on that can you convert 10 INR to USD")]

ai_message = llm_with_tools.invoke(messages)
messages.append(ai_message)

In [27]:
ai_message.tool_calls

[]

In [28]:
for tool_call in ai_message.tool_calls:
    if tool_call['name'] == 'get_conversion_factor':
        tool_message1 = get_conversion_factor.invoke(tool_call)
        conversion_rate = json.loads(tool_message1.content)['conversion_rate']
        print(conversion_rate)

In [29]:
llm_with_tools = llm.bind_tools([convert])

In [30]:
messages.append(AIMessage(content = f"The conversion rate between the currency is {conversion_rate}"))

In [ ]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD , and based on that can you convert 10 INR to USD', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Exchange rates fluctuate. I don’t have live data, but a typical recent rate (late 2024) is about:\n- 1 USD ≈ 83.5 INR, so 1 INR ≈ 0.01197 USD\n\nUsing that example rate:\n- 10 INR × 0.01197 USD/INR ≈ 0.1197 USD (about $0.12)\n\nIf you have a specific or up-to-date rate, share it and I’ll recompute precisely.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1329, 'prompt_tokens': 159, 'total_tokens': 1488, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 1216, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EXL0nNlToVIIBtc7K7j16qW3poWj5', 'service

In [34]:
response = llm_with_tools.invoke(messages).content

In [35]:
print(response)

Apologies for the conflicting figures earlier. Exchange rates vary, but a typical late-2024 rate is:
- 1 USD ≈ 83.5 INR → 1 INR ≈ 0.01197 USD

Using that:
- 10 INR × 0.01197 USD/INR ≈ $0.1197 (about $0.12)

If you prefer to use 1 INR = 0.01032 USD, then 10 INR = $0.1032. Let me know which rate you want to use (or provide a live rate) and I’ll compute precisely.
